# 2 · Memory tools — agent-controlled memory

`create_memory_tools` returns two tools the model can call deliberately:
`search_memories` and `add_memories`. The `user_id` scope is fixed by you,
never by the model, so an agent cannot read or write another user's memory.

These compose nicely with `EngramMiddleware` (automatic recall + explicit
save), but here we use them on their own.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.
> For the agent cells you also need `langchain-anthropic` (Option A installs it; for Option B run `%pip install langchain-anthropic`).

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".." langchain-anthropic

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY


def seed_memory(text, user_id, **kwargs):
    """Add a memory and block until the async pipeline commits it."""
    run = client.memories.add(text, user_id=user_id, **kwargs)
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(query=query, user_id=user_id, **kwargs)
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(query=query, user_id=user_id, **kwargs)

### Inspect the tools

In [ ]:
from langchain_engram import create_memory_tools

USER = 'demo-bob'
tools = create_memory_tools(user_id=USER)
for t in tools:
    print(t.name, '->', list(t.args))  # model controls only query/text

### Let an agent remember, then recall
Turn 1 asks the agent to remember something (it calls `add_memories`).
We wait for the pipeline, then a fresh turn recalls it via `search_memories`.

In [ ]:
from langchain.agents import create_agent

agent = create_agent('anthropic:claude-sonnet-4-6', tools=tools)

agent.invoke(
    {'messages': [{'role': 'user',
                   'content': 'Please remember that I live in Lisbon.'}]}
)

wait_until_searchable('where does the user live', USER, 'lisbon')

In [ ]:
out = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Where do I live?'}]}
)
print(out['messages'][-1].content)

### Call the tools directly (no LLM)
Useful for quick scripted testing.

In [ ]:
search_tool, add_tool = tools
print(add_tool.invoke({'text': 'The user prefers window seats.'}))
time.sleep(5)
print(search_tool.invoke({'query': 'seating preference'}))